In [9]:
%%writefile problem1_thread_tasks.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define N 1024

__global__ void differentTasks(int *input,
                                long long *output,
                                int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid == 0) {
        long long sum = 0;
        for (int i = 1; i <= n; i++) sum += i;
        output[0] = sum;
    } else if (tid == 1) {
        output[1] = (long long)n * (n + 1) / 2;
    } else if (tid < n) {
        output[tid] = input[tid];
    }
}

int main() {
    printf("===== Problem 1: Different Tasks =====\n");
    printf("N = %d\n\n", N);

    int       *h_input  = (int*)malloc(N*sizeof(int));
    long long *h_output = (long long*)malloc(N*sizeof(long long));
    for (int i = 0; i < N; i++) h_input[i] = i + 1;

    int       *d_input;
    long long *d_output;
    cudaMalloc(&d_input,  N*sizeof(int));
    cudaMalloc(&d_output, N*sizeof(long long));
    cudaMemset(d_output, 0, N*sizeof(long long));
    cudaMemcpy(d_input, h_input,
               N*sizeof(int), cudaMemcpyHostToDevice);

    printf("Block size: %d threads\n", N);
    printf("Grid  size: 1 block\n\n");

    differentTasks<<<1, N>>>(d_input, d_output, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_output, d_output,
               N*sizeof(long long), cudaMemcpyDeviceToHost);

    long long expected = (long long)N*(N+1)/2;

    printf("--- Task A (Thread 0): Iterative Sum ---\n");
    printf("  Result   : %lld\n", h_output[0]);
    printf("  Expected : %lld\n", expected);
    printf("  Match    : %s\n\n",
           h_output[0]==expected ? "YES" : "NO");

    printf("--- Task B (Thread 1): Formula Sum ---\n");
    printf("  Result   : %lld\n", h_output[1]);
    printf("  Expected : %lld\n", expected);
    printf("  Match    : %s\n\n",
           h_output[1]==expected ? "YES" : "NO");

    printf("Both threads agree: %s\n",
           h_output[0]==h_output[1] ? "YES" : "NO");

    cudaFree(d_input); cudaFree(d_output);
    free(h_input); free(h_output);
    return 0;
}

Overwriting problem1_thread_tasks.cu


In [10]:
!nvcc problem1_thread_tasks.cu -o problem1 && ./problem1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Problem 1: Different Tasks =====
N = 1024

Block size: 1024 threads
Grid  size: 1 block

--- Task A (Thread 0): Iterative Sum ---
  Result   : 524800
  Expected : 524800
  Match    : YES

--- Task B (Thread 1): Formula Sum ---
  Result   : 524800
  Expected : 524800
  Match    : YES

Both threads agree: YES


In [11]:
%%writefile problem2_merge_sort.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define ARRAY_SIZE 1000

// ── CPU Merge Sort ────────────────────────────────────────
void merge_cpu(int *arr, int *temp,
               int left, int mid, int right) {
    int i=left, j=mid+1, k=left;
    while (i<=mid && j<=right)
        temp[k++]=(arr[i]<=arr[j])?arr[i++]:arr[j++];
    while (i<=mid)   temp[k++]=arr[i++];
    while (j<=right) temp[k++]=arr[j++];
    for (int x=left; x<=right; x++) arr[x]=temp[x];
}

void mergeSort_cpu(int *arr, int *temp,
                   int left, int right) {
    if (left>=right) return;
    int mid=left+(right-left)/2;
    mergeSort_cpu(arr, temp, left, mid);
    mergeSort_cpu(arr, temp, mid+1, right);
    merge_cpu(arr, temp, left, mid, right);
}

// ── GPU Merge Sort ────────────────────────────────────────
__device__ void merge_gpu(int *arr, int *temp,
                           int left, int mid, int right) {
    int i=left, j=mid+1, k=left;
    while (i<=mid && j<=right)
        temp[k++]=(arr[i]<=arr[j])?arr[i++]:arr[j++];
    while (i<=mid)   temp[k++]=arr[i++];
    while (j<=right) temp[k++]=arr[j++];
    for (int x=left; x<=right; x++) arr[x]=temp[x];
}

__global__ void mergeSortKernel(int *arr, int *temp,
                                 int n, int width) {
    int tid  = blockIdx.x*blockDim.x+threadIdx.x;
    int left = tid*2*width;
    if (left>=n) return;
    int mid  = min(left+width-1, n-1);
    int right= min(left+2*width-1, n-1);
    if (mid<right) merge_gpu(arr,temp,left,mid,right);
}

// ── Helpers ───────────────────────────────────────────────
void fill_random(int *arr, int n, unsigned int seed) {
    srand(seed);
    for (int i=0; i<n; i++) arr[i]=rand()%10000;
}

int is_sorted(int *arr, int n) {
    for (int i=0; i<n-1; i++)
        if (arr[i]>arr[i+1]) return 0;
    return 1;
}

double get_ms(struct timespec s, struct timespec e) {
    return (e.tv_sec-s.tv_sec)*1000.0
          +(e.tv_nsec-s.tv_nsec)/1e6;
}

// ── Main ──────────────────────────────────────────────────
int main() {
    printf("===== Problem 2: Merge Sort =====\n");
    printf("Array size: %d\n\n", ARRAY_SIZE);

    int *arr_cpu =(int*)malloc(ARRAY_SIZE*sizeof(int));
    int *temp_cpu=(int*)malloc(ARRAY_SIZE*sizeof(int));
    int *arr_gpu =(int*)malloc(ARRAY_SIZE*sizeof(int));

    // Part A: CPU
    fill_random(arr_cpu, ARRAY_SIZE, 42);
    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    mergeSort_cpu(arr_cpu, temp_cpu, 0, ARRAY_SIZE-1);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    double cpu_time = get_ms(t0, t1);

    printf("--- Part A: CPU Pipelined Merge Sort ---\n");
    printf("  Time   : %.4f ms\n", cpu_time);
    printf("  Sorted : %s\n\n",
           is_sorted(arr_cpu,ARRAY_SIZE)?"YES":"NO");

    // Part B: GPU
    fill_random(arr_gpu, ARRAY_SIZE, 42);
    int *d_arr, *d_temp;
    cudaMalloc(&d_arr,  ARRAY_SIZE*sizeof(int));
    cudaMalloc(&d_temp, ARRAY_SIZE*sizeof(int));
    cudaMemcpy(d_arr, arr_gpu,
               ARRAY_SIZE*sizeof(int),
               cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    for (int width=1; width<ARRAY_SIZE; width*=2) {
        int merges=(ARRAY_SIZE+2*width-1)/(2*width);
        int blocks=(merges+127)/128;
        mergeSortKernel<<<blocks,128>>>(
            d_arr, d_temp, ARRAY_SIZE, width);
        cudaDeviceSynchronize();
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float gpu_time=0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(arr_gpu, d_arr,
               ARRAY_SIZE*sizeof(int),
               cudaMemcpyDeviceToHost);

    printf("--- Part B: CUDA Parallel Merge Sort ---\n");
    printf("  Time   : %.4f ms\n", gpu_time);
    printf("  Sorted : %s\n\n",
           is_sorted(arr_gpu,ARRAY_SIZE)?"YES":"NO");

    // Part C: Comparison
    printf("--- Part C: Performance Comparison ---\n");
    printf("  CPU time : %.4f ms\n", cpu_time);
    printf("  GPU time : %.4f ms\n", gpu_time);
    if (gpu_time<cpu_time)
        printf("  Speedup  : %.2fx (GPU faster)\n",
               cpu_time/gpu_time);
    else
        printf("  Slowdown : %.2fx (CPU faster)\n",
               gpu_time/cpu_time);
    printf("  NOTE: GPU wins at N >= 100,000+\n");

    int match=1;
    for (int i=0; i<ARRAY_SIZE; i++)
        if (arr_cpu[i]!=arr_gpu[i]){match=0;break;}
    printf("  Outputs match: %s\n\n",
           match?"YES":"NO");

    printf("--- Sample (first 10 sorted) ---\n");
    printf("CPU: ");
    for (int i=0;i<10;i++) printf("%d ",arr_cpu[i]);
    printf("\nGPU: ");
    for (int i=0;i<10;i++) printf("%d ",arr_gpu[i]);
    printf("\n");

    cudaFree(d_arr); cudaFree(d_temp);
    free(arr_cpu); free(temp_cpu); free(arr_gpu);
    return 0;
}

Overwriting problem2_merge_sort.cu


In [12]:
!nvcc problem2_merge_sort.cu -o problem2 && ./problem2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Problem 2: Merge Sort =====
Array size: 1000

--- Part A: CPU Pipelined Merge Sort ---
  Time   : 0.1283 ms
  Sorted : YES

--- Part B: CUDA Parallel Merge Sort ---
  Time   : 0.7317 ms
  Sorted : YES

--- Part C: Performance Comparison ---
  CPU time : 0.1283 ms
  GPU time : 0.7317 ms
  Slowdown : 5.71x (CPU faster)
  NOTE: GPU wins at N >= 100,000+
  Outputs match: YES

--- Sample (first 10 sorted) ---
CPU: 18 26 46 51 58 61 81 86 90 111 
GPU: 18 26 46 51 58 61 81 86 90 111 


In [13]:
%%writefile problem3_vector_add.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define VECTOR_SIZE (1 << 20)
#define BLOCK_SIZE  256

__global__ void vectorAdd(float *A, float *B,
                          float *C, int n) {
    int i = blockIdx.x*blockDim.x+threadIdx.x;
    if (i<n) C[i] = A[i]+B[i];
}

int main() {
    printf("===== Problem 3: Vector Addition =====\n\n");

    int    n     = VECTOR_SIZE;
    size_t bytes = n*sizeof(float);

    float *h_A=(float*)malloc(bytes);
    float *h_B=(float*)malloc(bytes);
    float *h_C=(float*)malloc(bytes);

    for (int i=0; i<n; i++) {
        h_A[i]=i*0.5f;
        h_B[i]=i*1.5f;
    }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);
    cudaMemcpy(d_A,h_A,bytes,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,bytes,cudaMemcpyHostToDevice);

    int gridSize=(n+BLOCK_SIZE-1)/BLOCK_SIZE;

    printf("Vector size : %d elements\n", n);
    printf("Block size  : %d threads\n", BLOCK_SIZE);
    printf("Grid  size  : %d blocks\n\n", gridSize);

    // 1.3 Theoretical BW
    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);

    printf("--- Device Properties ---\n");
    printf("  GPU               : %s\n", prop.name);
    printf("  Memory Clock Rate : %d kHz\n",
           prop.memoryClockRate);
    printf("  Memory Bus Width  : %d bits\n\n",
           prop.memoryBusWidth);

    double theoreticalBW =
        2.0*prop.memoryClockRate*1000.0
        *(prop.memoryBusWidth/8.0)/1.0e9;

    printf("--- 1.3 Theoretical Bandwidth ---\n");
    printf("  Formula: 2 x memClockRate(Hz) x "
           "busWidth(bytes) / 1e9\n");
    printf("  = 2 x %d x 1000 x (%d/8) / 1e9\n",
           prop.memoryClockRate,
           prop.memoryBusWidth);
    printf("  Theoretical BW : %.2f GB/s\n\n",
           theoreticalBW);

    // 1.2 Timing
    cudaEvent_t evStart, evStop;
    cudaEventCreate(&evStart);
    cudaEventCreate(&evStop);
    cudaEventRecord(evStart);

    vectorAdd<<<gridSize,BLOCK_SIZE>>>(d_A,d_B,d_C,n);

    cudaEventRecord(evStop);
    cudaEventSynchronize(evStop);
    float kernel_ms=0.0f;
    cudaEventElapsedTime(&kernel_ms,evStart,evStop);

    printf("--- 1.2 Kernel Timing ---\n");
    printf("  Kernel time : %.4f ms\n\n", kernel_ms);

    // 1.4 Measured BW
    long long RBytes    = 2LL*n*sizeof(float);
    long long WBytes    = 1LL*n*sizeof(float);
    long long totalB    = RBytes+WBytes;
    double    time_s    = kernel_ms/1000.0;
    double    measuredBW= totalB/time_s/1.0e9;

    printf("--- 1.4 Measured Bandwidth ---\n");
    printf("  Bytes Read    : %lld bytes (%.2f MB)\n",
           RBytes, RBytes/1.0e6);
    printf("  Bytes Written : %lld bytes (%.2f MB)\n",
           WBytes, WBytes/1.0e6);
    printf("  Total Bytes   : %lld bytes (%.2f MB)\n",
           totalB, totalB/1.0e6);
    printf("  Kernel Time   : %.6f s\n", time_s);
    printf("  Measured BW   : %.2f GB/s\n\n",
           measuredBW);

    printf("========================================\n");
    printf("  Theoretical BW : %8.2f GB/s\n",
           theoreticalBW);
    printf("  Measured BW    : %8.2f GB/s\n",
           measuredBW);
    printf("  Efficiency     : %8.2f %%\n",
           (measuredBW/theoreticalBW)*100.0);
    printf("========================================\n\n");

    // Verify
    cudaMemcpy(h_C,d_C,bytes,cudaMemcpyDeviceToHost);
    int errors=0;
    for (int i=0; i<n; i++)
        if (fabsf(h_C[i]-(h_A[i]+h_B[i]))>1e-4f)
            errors++;
    printf("Verification : %s (errors: %d)\n",
           errors==0?"PASS":"FAIL", errors);

    printf("\n--- Sample Results ---\n");
    printf("%-6s %-10s %-10s %-10s\n",
           "i","A[i]","B[i]","C[i]");
    for (int i=0; i<5; i++)
        printf("%-6d %-10.1f %-10.1f %-10.1f\n",
               i, h_A[i], h_B[i], h_C[i]);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    cudaEventDestroy(evStart);
    cudaEventDestroy(evStop);
    return 0;
}

Overwriting problem3_vector_add.cu


In [14]:
!nvcc -O3 problem3_vector_add.cu -o problem3 && ./problem3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
===== Problem 3: Vector Addition =====

Vector size : 1048576 elements
Block size  : 256 threads
Grid  size  : 4096 blocks

--- Device Properties ---
  GPU               : Tesla T4
  Memory Clock Rate : 5001000 kHz
  Memory Bus Width  : 256 bits

--- 1.3 Theoretical Bandwidth ---
  Formula: 2 x memClockRate(Hz) x busWidth(bytes) / 1e9
  = 2 x 5001000 x 1000 x (256/8) / 1e9
  Theoretical BW : 320.06 GB/s

--- 1.2 Kernel Timing ---
  Kernel time : 0.2109 ms

--- 1.4 Measured Bandwidth ---
  Bytes Read    : 8388608 bytes (8.39 MB)
  Bytes Written : 4194304 bytes (4.19 MB)
  Total Bytes   : 12582912 bytes (12.58 MB)
  Kernel Time   : 0.000211 s
  Measured BW   : 59.66 GB/s

  Theoretical BW :   320.06 GB/s
  Measured BW    :    59.66 GB/s
  Efficiency     :    18.64 %

Verification : PASS (errors: 0)

--- 